In [1]:
#!pip install -U torch scikit-learn torchvision timm

# Baseline model for: Trustworthy Deep Learning for Chest X-ray Disease Detection
## Stage 1 - Baseline Establishment
 
Architecture : DenseNet121 (ImageNet-pretrained, timm), 4-class head

Dataset      : COVID-19 Radiography Database (Kaggle) - COVID / Normal / Lung_Opacity / Viral Pneumonia

Split        : 70/15/15 stratified, fixed seed

Preprocessing: 224x224, 3-channel, ImageNet normalization

Augmentation : random rotation, horizontal flip, brightness/contrast jitter, random crop (train only)

Optimizer    : AdamW (SGD-momentum / Adam selectable for the hyperparameter sweep)

Loss         : Cross-entropy with class weighting

Training     : two-phase transfer learning (frozen backbone -> fine-tune final dense blocks)
              + early stopping on validation loss

Evaluation   : accuracy, precision, recall, F1 (per-class + macro), ROC-AUC, confusion matrix
 
Run on Google Colab / Kaggle (free GPU). Example:
    python baseline_densenet121.py --data_dir /kaggle/input/covid19-radiography-database \
        --output_dir ./runs/densenet121_baseline


In [2]:
import argparse
import copy
import json
import os
import random
from pathlib import Path
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score, accuracy_score

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
 
try:
    import timm
except ImportError as e:
    raise ImportError(
        "This script requires `timm`. Install with: pip install timm --break-system-packages"
    ) from e

## Reproducibility

In [3]:
def set_seed(seed: int = 42):
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [5]:
def build_transforms(img_size: int = 224):
  train_tf = transforms.Compose(
    [
      transforms.Resize((img_size, img_size)),
      transforms.RandomCrop(img_size, padding=8, padding_mode='reflect'),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(degrees=10),
      transforms.ColorJitter(brightness=0.2, contrast=0.2),
      transforms.Grayscale(num_output_channels=3),
      transforms.ToTensor(),
      transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ]
  )

  eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
  ])
  return train_tf, eval_tf

In [6]:
class TransformSubset(Dataset):
  """Wraps a Subset of an ImageFolder so train/val/test can each use a different transform."""

  def __init__(self, subset: Subset, transform):
    self.subset = subset
    self.transform = transform

  def __len__(self):
    return len(self.subset)

  def __getitem__(self, idx):
    img, label = self.subset[idx]
    if self.transform is not None:
      img = self.transform(img)
    return img, label

In [7]:
def stratified_split(dataset: ImageFolder, seed: int = 42):
  """70/15/15 stratified split of dataset into train, val, and test subsets."""

  targets = np.array(dataset.targets)
  indices = np.arange(len(dataset))

  train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=targets, random_state=seed
  )

  val_idx, test_idx = train_test_split(
    temp_idx,
    test_size= 0.50,
    stratify=targets[temp_idx],
    random_state = seed,
  )

  return train_idx, val_idx, test_idx

In [8]:
def build_dataloaders(data_dir: str, img_size: int, batch_size: int, seed: int, num_workers: int = 4):
  # ImageFolder expects: data_dir/<class_name>/*.png
  # Load once without transform so PIL images can be transformed differently per split.
  def only_images_folder(path):
    p = Path(path)
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif"}
    return p.parent.name.lower() == "images" and p.suffix.lower() in valid_exts
  base_dataset = ImageFolder(root=data_dir, is_valid_file=only_images_folder)

  train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=seed)
  train_tf, eval_tf = build_transforms(img_size)

  train_ds = TransformSubset(Subset(base_dataset, train_idx), train_tf)
  val_ds = TransformSubset(Subset(base_dataset, val_idx), eval_tf)
  test_ds = TransformSubset(Subset(base_dataset, test_idx), eval_tf)

  pin_memory = torch.cuda.is_available()
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

  class_names = base_dataset.classes
  train_targets = np.array(base_dataset.targets)[train_idx]
  datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
  return train_loader, val_loader, test_loader, class_names, train_targets, datasets

In [9]:
def compute_class_weights(train_targets: np.ndarray, num_classes: int) -> torch.Tensor:
  counts = np.bincount(train_targets, minlength=num_classes).astype(np.float32)
  counts[counts == 0] = 1.0  # avoid div-by-zero
  weights = counts.sum() / (num_classes * counts)
  return torch.tensor(weights, dtype=torch.float32)

## Model

In [10]:
def build_model(num_classes: int = 4) -> nn.Module:
    model = timm.create_model("densenet121", pretrained=True, num_classes=num_classes)
    return model

In [11]:
def freeze_backbone(model: nn.Module):
    """Phase 1: freeze everything except the final classifier head."""
    for name, param in model.named_parameters():
        if "classifier" in name or "fc" in name:  # timm densenet head is named 'classifier'
            param.requires_grad = True
        else:
            param.requires_grad = False

In [12]:
def unfreeze_final_blocks(model: nn.Module, num_dense_blocks_to_unfreeze: int = 1):
    """Phase 2: unfreeze the classifier + the last N dense blocks for end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
    # timm densenet121 feature blocks are named features.denseblock1..4 / features.norm5
    unfreeze_names = ["features.norm5"] + [
        f"features.denseblock{4 - i}" for i in range(num_dense_blocks_to_unfreeze)
    ] + [
        f"features.transition{3 - i}" for i in range(num_dense_blocks_to_unfreeze)
    ]
    for name, param in model.named_parameters():
        if any(name.startswith(u) for u in unfreeze_names):
            param.requires_grad = True

In [13]:
def build_optimizer(model, name: str, lr: float, weight_decay: float):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

In [14]:
def build_scheduler(optimizer, name: str, epochs: int):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {name}")

## Training / evaluation loops

In [15]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool, desc: str = ""):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
 
    non_blocking = device.type == "cuda"
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
 
            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(loss=f"{total_loss / total:.4f}", acc=f"{correct / total:.4f}")
 
    return total_loss / total, correct / total

In [16]:
def train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, phase_name, output_dir,
):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []
 
    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, desc=train_desc
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, desc=val_desc
        )
 
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
 
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break
 
    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

In [17]:
@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
 
    non_blocking = device.type == "cuda"
    for images, labels in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
 
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
 
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
 
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)
 
    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None
 
    results = {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_auc_macro": auc_macro,
        "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
        "class_names": class_names,
    }
 
    with open(Path(output_dir) / "test_results.json", "w") as f:
        json.dump(results, f, indent=2)
 
    print("\n=== Test set performance ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")

    # --- Save confusion matrix as CSV ---
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / "confusion_matrix.csv")

    # --- Save one-row summary CSV (accuracy, macro/weighted f1 & recall, ROC-AUC) ---
    summary = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_precision": report["macro avg"]["precision"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "weighted_recall": report["weighted avg"]["recall"],
        "roc_auc_macro": auc_macro,
    }
    pd.DataFrame([summary]).to_csv(Path(output_dir) / "summary_metrics.csv", index=False)
    
 
    return results

In [18]:
parser = argparse.ArgumentParser(description="DenseNet121 baseline - Stage 1")
parser.add_argument("--data_dir", type=str,
                    default=r"/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset",
                    help="Path to dataset root with one subfolder per class")
parser.add_argument("--output_dir", type=str, default="./runs/densenet121_baseline")
parser.add_argument("--img_size", type=int, default=224)
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--num_workers", type=int, default=4)
parser.add_argument("--device", type=str, default="auto", choices=["auto", "cuda", "cpu"],
                     help="'auto' picks CUDA if available, else CPU. Force 'cpu' to sanity-check "
                          "the pipeline locally before running full training on a GPU box.")

# Phase 1 (frozen backbone, train head)
parser.add_argument("--phase1_epochs", type=int, default=15)
parser.add_argument("--phase1_lr", type=float, default=1e-3)

# Phase 2 (fine-tune final dense blocks)
parser.add_argument("--phase2_epochs", type=int, default=40)
parser.add_argument("--phase2_lr", type=float, default=1e-5)
parser.add_argument("--unfreeze_blocks", type=int, default=1)

# Hyperparameter-sweep knobs
parser.add_argument("--optimizer", type=str, default="adamw", choices=["adamw", "adam", "sgd"])
parser.add_argument("--weight_decay", type=float, default=1e-4)
parser.add_argument("--scheduler", type=str, default="cosine", choices=["cosine", "step", "plateau", "none"])
parser.add_argument("--patience", type=int, default=5)

_StoreAction(option_strings=['--patience'], dest='patience', nargs=None, const=None, default=5, type=<class 'int'>, choices=None, required=False, help=None, metavar=None)

In [19]:
args, _ = parser.parse_known_args()

Path(args.output_dir).mkdir(parents=True, exist_ok=True)
set_seed(args.seed)

In [20]:
if args.device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("--device cuda was requested but no CUDA GPU is available.")
if args.device == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(args.device)

print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(device)}")
    print(f"  CUDA version: {torch.version.cuda}")
    total_mem_gb = torch.cuda.get_device_properties(device).total_memory / (1024 ** 3)
    print(f"  GPU memory: {total_mem_gb:.1f} GB")
else:
    print("  Running on CPU - training DenseNet121 here will be much slower; "
          "use --device cpu only for a quick smoke test with a small --batch_size.")


Using device: cuda
  GPU: Tesla T4
  CUDA version: 12.8
  GPU memory: 14.6 GB


In [21]:
# ---- Data ----
train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    args.data_dir, args.img_size, args.batch_size, args.seed, args.num_workers
)
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")
print(f"Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")

class_weights = compute_class_weights(train_targets, num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Train/Val/Test sizes: 14815/3175/3175


## Cache preprocessed images (optional)

Writes the fully preprocessed (resize + grayscale + normalize) images to disk as PNGs, one folder per split/class, so the expensive PIL decode/resize doesn't have to be repeated on every epoch or run.

Note: the train split's random augmentation (crop/flip/rotation/jitter) is intentionally **not** baked into this cache — that transform is meant to produce a different result each epoch, so caching one fixed draw of it would silently remove that variability. The train images below are cached with the same deterministic transform used for val/test.

In [22]:
def cache_preprocessed_images(dataset, class_names, split_name: str, output_dir, mean=IMAGENET_MEAN, std=IMAGENET_STD):
  """Run every sample in `dataset` through its transform once and save the result as a PNG.

  Denormalizes back to [0, 1] before saving so the cached files are viewable images
  rather than raw normalized tensors.
  """
  split_dir = Path(output_dir) / split_name
  for cls in class_names:
    (split_dir / cls).mkdir(parents=True, exist_ok=True)

  mean_t = torch.tensor(mean).view(3, 1, 1)
  std_t = torch.tensor(std).view(3, 1, 1)
  to_pil = transforms.ToPILImage()
  counts = {cls: 0 for cls in class_names}

  for img_tensor, label in tqdm(dataset, desc=f"Caching {split_name}", dynamic_ncols=True):
    cls = class_names[label]
    denorm = (img_tensor * std_t + mean_t).clamp(0, 1)
    to_pil(denorm).save(split_dir / cls / f"{counts[cls]:05d}.png")
    counts[cls] += 1

  print(f"[{split_name}] cached {sum(counts.values())} images to {split_dir}")
  return counts

In [23]:
# Deterministic (eval-style) view of the train split — same underlying samples, no random augmentation.
train_ds_deterministic = TransformSubset(datasets["train"].subset, datasets["val"].transform)

cache_dir = Path(args.output_dir) / "preprocessed_images"
cache_preprocessed_images(train_ds_deterministic, class_names, "train", cache_dir)
cache_preprocessed_images(datasets["val"], class_names, "val", cache_dir)
cache_preprocessed_images(datasets["test"], class_names, "test", cache_dir)

Caching train: 100%|██████████| 14815/14815 [05:46<00:00, 42.76it/s]


[train] cached 14815 images to runs/densenet121_baseline/preprocessed_images/train


Caching val: 100%|██████████| 3175/3175 [01:18<00:00, 40.38it/s]


[val] cached 3175 images to runs/densenet121_baseline/preprocessed_images/val


Caching test: 100%|██████████| 3175/3175 [01:18<00:00, 40.37it/s]

[test] cached 3175 images to runs/densenet121_baseline/preprocessed_images/test


{'COVID': 542, 'Lung_Opacity': 902, 'Normal': 1529, 'Viral Pneumonia': 202}

In [24]:
model = build_model(num_classes=num_classes).to(device)

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

In [25]:
freeze_backbone(model)
opt1 = build_optimizer(model, args.optimizer, args.phase1_lr, args.weight_decay)
sched1 = build_scheduler(opt1, args.scheduler, args.phase1_epochs)
model = train_phase(
    model, train_loader, val_loader, criterion, opt1, sched1,
    device, args.phase1_epochs, args.patience, "phase1_frozen", args.output_dir,
)

[phase1_frozen] epoch 1/15 train_loss=0.6507 train_acc=0.7216 val_loss=0.4934 val_acc=0.8088


[phase1_frozen] epoch 2/15 train_loss=0.4753 train_acc=0.7919 val_loss=0.4046 val_acc=0.8255


[phase1_frozen] epoch 3/15 train_loss=0.4342 train_acc=0.8076 val_loss=0.3807 val_acc=0.8409


[phase1_frozen] epoch 4/15 train_loss=0.4083 train_acc=0.8193 val_loss=0.3885 val_acc=0.8422


[phase1_frozen] epoch 5/15 train_loss=0.3881 train_acc=0.8291 val_loss=0.3535 val_acc=0.8513


...

[phase1_frozen] epoch 11/15 train_loss=0.3598 train_acc=0.8400 val_loss=0.3384 val_acc=0.8611


[phase1_frozen] epoch 12/15 train_loss=0.3534 train_acc=0.8400 val_loss=0.3376 val_acc=0.8605


[phase1_frozen] epoch 13/15 train_loss=0.3536 train_acc=0.8414 val_loss=0.3383 val_acc=0.8561


[phase1_frozen] epoch 14/15 train_loss=0.3584 train_acc=0.8409 val_loss=0.3344 val_acc=0.8494


[phase1_frozen] epoch 15/15 train_loss=0.3512 train_acc=0.8423 val_loss=0.3336 val_acc=0.8564


In [26]:
unfreeze_final_blocks(model, num_dense_blocks_to_unfreeze=args.unfreeze_blocks)
opt2 = build_optimizer(model, args.optimizer, args.phase2_lr, args.weight_decay)
sched2 = build_scheduler(opt2, args.scheduler, args.phase2_epochs)
model = train_phase(
    model, train_loader, val_loader, criterion, opt2, sched2,
    device, args.phase2_epochs, args.patience, "phase2_finetune", args.output_dir,
)

[phase2_finetune] epoch 1/40 train_loss=0.3504 train_acc=0.8433 val_loss=0.3235 val_acc=0.8671


[phase2_finetune] epoch 2/40 train_loss=0.3328 train_acc=0.8506 val_loss=0.3069 val_acc=0.8709


[phase2_finetune] epoch 3/40 train_loss=0.3054 train_acc=0.8624 val_loss=0.2935 val_acc=0.8712


[phase2_finetune] epoch 4/40 train_loss=0.2979 train_acc=0.8655 val_loss=0.2814 val_acc=0.8778


[phase2_finetune] epoch 5/40 train_loss=0.2848 train_acc=0.8702 val_loss=0.2724 val_acc=0.8809


...

[phase2_finetune] epoch 33/40 train_loss=0.1902 train_acc=0.9127 val_loss=0.2083 val_acc=0.9109


[phase2_finetune] epoch 34/40 train_loss=0.1949 train_acc=0.9099 val_loss=0.2023 val_acc=0.9121


[phase2_finetune] epoch 35/40 train_loss=0.1925 train_acc=0.9120 val_loss=0.2051 val_acc=0.9109


[phase2_finetune] epoch 36/40 train_loss=0.1927 train_acc=0.9115 val_loss=0.2072 val_acc=0.9112


[phase2_finetune] epoch 37/40 train_loss=0.1918 train_acc=0.9120 val_loss=0.2048 val_acc=0.9115
[phase2_finetune] Early stopping at epoch 37 (no improvement for 5 epochs).


In [27]:
ckpt_path = Path(args.output_dir) / "densenet121_baseline.pt"
torch.save({"model_state_dict": model.state_dict(), "class_names": class_names}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

Saved checkpoint to runs/densenet121_baseline/densenet121_baseline.pt


In [28]:
# ---- Evaluate on held-out test set ----
evaluate(model, test_loader, class_names, device, args.output_dir)


=== Test set performance ===
Test accuracy: 0.9150
                 precision    recall  f1-score   support

          COVID     0.9107    0.9225    0.9166       542
   Lung_Opacity     0.8920    0.8969    0.8944       902
         Normal     0.9310    0.9176    0.9242      1529
Viral Pneumonia     0.9104    0.9554    0.9324       202

       accuracy                         0.9150      3175
      macro avg     0.9110    0.9231    0.9169      3175
   weighted avg     0.9151    0.9150    0.9150      3175

Confusion matrix:
 [[ 500   20   19    3]
 [  15  809   77    1]
 [  33   78 1403   15]
 [   1    0    8  193]]
Macro ROC-AUC: 0.9870


{'classification_report': {'COVID': {'precision': 0.9107468123861566,
   'recall': 0.922509225092251,
   'f1-score': 0.916590284142988,
   'support': 542.0},
  'Lung_Opacity': {'precision': 0.8919514884233738,
   'recall': 0.8968957871396895,
   'f1-score': 0.8944168048645661,
   'support': 902.0},
  'Normal': {'precision': 0.9309887193098872,
   'recall': 0.9175931981687377,
   'f1-score': 0.9242424242424242,
   'support': 1529.0},
  'Viral Pneumonia': {'precision': 0.910377358490566,
   'recall': 0.9554455445544554,
   'f1-score': 0.9323671497584541,
   'support': 202.0},
  'accuracy': 0.9149606299212598,
  'macro avg': {'precision': 0.911016094652496,
   'recall': 0.9231109387387834,
   'f1-score': 0.9169041657521082,
   'support': 3175.0},
  'weighted avg': {'precision': 0.9151316513735723,
   'recall': 0.9149606299212598,
   'f1-score': 0.9149797552476259,
   'support': 3175.0}},
 'confusion_matrix': [[500, 20, 19, 3],
  [15, 809, 77, 1],
  [33, 78, 1403, 15],
  [1, 0, 8, 193]],
 